In [2]:
# Correlations between LV1 EEG scores and clinical variables in ASD + supplementarfy figure 1
# We correlate the X_LV1 (composite EEG score) with SRS, SSP_total, and CSS_total in ASD subjects only
# Each correlation uses all ASD subjects who have that specific variable
# (no subjects dropped due to missing data in other variables)
# FDR correction applied across three correlations

# Step 1 : import data and set paths

import pandas as pd, numpy as np, os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from statsmodels.stats.multitest import multipletests

analysis_name = "analysis_october"

out_path     = rf'C:\Users\cigde\Dropbox\Habituation_paper\October_Emily\pls_regression\plsdo_outputs\{analysis_name}'
in_path      = rf'C:\Users\cigde\Dropbox\Habituation_paper\October_Emily\pls_regression\plsdo_inputs\{analysis_name}'
master_path  = r'C:\Users\cigde\Dropbox\Habituation_paper\October_Emily\input_arrays\pls_regression'


In [3]:
# Step 2: rebuild df (scores + group), then asd_scores (+ clinical)
scores = pd.read_csv(os.path.join(out_path, 'data', 'subject_scores.csv'), dtype={'UniqueID': str})
parts  = pd.read_csv(os.path.join(in_path, f'participants_{analysis_name}.csv'), dtype={'UniqueID': str})
df = scores.merge(parts[['UniqueID', 'group_original']], on='UniqueID', how='left')
master = pd.read_csv(os.path.join(master_path, 'master_RS.csv'), dtype={'UniqueID': str})
asd_scores = df[df['group_original'] == 'ASD'].merge(
    master[['UniqueID', 'SRS_tscore', 'SSP_total', 'CSS_total']], on='UniqueID', how='left')

# Check: ASD table after merge
print(asd_scores.shape)
print(asd_scores.head())

(166, 7)
       UniqueID     X_LV1     Y_LV1 group_original  SRS_tscore  SSP_total  \
0  104324981539  0.717386 -1.242410            ASD        74.0      123.0   
1  109182897329  0.049434  0.795026            ASD        63.0        NaN   
2  112353982965 -2.787819  0.000501            ASD        90.0       91.0   
3  113902830642  1.677050 -1.325954            ASD         NaN        NaN   
4  118194667229 -2.743257 -0.697452            ASD        69.0      117.0   

   CSS_total  
0        7.0  
1        7.0  
2        6.0  
3        NaN  
4        7.0  


In [4]:
# Step 3: Correlate X_LV1 with each clinical variable in ASD, FDR corrected

clinical_vars = ['SRS_tscore', 'SSP_total', 'CSS_total']

results = []
for var in clinical_vars:
    # Drop subjects missing this variable
    subset = asd_scores[['X_LV1', var]].dropna()
    r, p = pearsonr(subset['X_LV1'], subset[var])
    results.append({'variable': var, 'r': r, 'p': p, 'n': len(subset)})

results_df = pd.DataFrame(results)


print(f"ASD subjects: {len(asd_scores)}")
print("\nAvailable N per clinical variable:")
print(asd_scores[['SRS_tscore', 'SSP_total', 'CSS_total']].notna().sum())

# FDR correction across three correlations
rejected, p_fdr, _, _ = multipletests(results_df['p'], method='fdr_bh')
results_df['p_fdr'] = p_fdr
results_df['significant'] = rejected

print(results_df.to_string(index=False))

fig_path     = r'C:\Users\cigde\Dropbox\Habituation_paper\October_Emily\final_figures\supplementary\suppl_fig_2'
os.makedirs(fig_path, exist_ok=True)
results_df.to_csv(os.path.join(fig_path, 'figure_suppl_2_correlations.csv'), index=False)

ASD subjects: 166

Available N per clinical variable:
SRS_tscore    131
SSP_total      95
CSS_total     142
dtype: int64
  variable         r        p   n    p_fdr  significant
SRS_tscore  0.186151 0.033269 131 0.049904         True
 SSP_total -0.139186 0.178556  95 0.178556        False
 CSS_total -0.197247 0.018630 142 0.049904         True


In [5]:
subset = asd_scores[['SRS_tscore', 'CSS_total']].dropna()
r, p = pearsonr(subset['SRS_tscore'], subset['CSS_total'])
print(f"r = {r:.3f}, p = {p:.4f}, n = {len(subset)}")

r = 0.275, p = 0.0029, n = 115


In [ ]:
#  Step 4: generate the figure and add the regression line

# Step 4.a: Select the variables that came back significant in Step 3

sig_vars = results_df.loc[results_df['significant'], 'variable'].tolist()
print(f"Significant variables: {sig_vars}")

# Step 4.b: Step 4b: Helper that returns the regression line and its 95% CI band

def regression_ci(x, y, x_line, ci=0.95):
    n = len(x); m, b = np.polyfit(x, y, 1)
    resid = y - (m*x + b)
    se = np.sqrt(np.sum(resid**2) / (n - 2))
    t_val = t.ppf((1 + ci) / 2, df=n - 2)
    se_line = se * np.sqrt(1/n + (x_line - x.mean())**2 / np.sum((x - x.mean())**2))
    return m*x_line + b, se_line * t_val

# Step 4c: One panel per significant variable: scatter, regression line and 95% CI

fig, axes = plt.subplots(1, len(sig_vars), figsize=(6*len(sig_vars), 5), squeeze=False)

for ax, var in zip(axes[0], sig_vars):
    subset = asd_scores[['X_LV1', var]].dropna()
    x_arr, y_arr = subset['X_LV1'].values, subset[var].values
    ax.scatter(x_arr, y_arr, color='red', alpha=0.5, s=30, edgecolors='black', linewidths=0.4, zorder=3)
    x_line = np.linspace(x_arr.min(), x_arr.max(), 200)
    y_line, ci_band = regression_ci(x_arr, y_arr, x_line)
    ax.plot(x_line, y_line, color='black', linestyle='--', linewidth=1.5)
    ax.fill_between(x_line, y_line - ci_band, y_line + ci_band, color='red', alpha=0.15)
    ax.set_xlabel('LV1 EEG Scores', fontsize=16)
    ax.set_ylabel(var, fontsize=16)
    ax.set_title(f'LV1 EEG scores vs {var}', fontsize=18)
    ax.tick_params(axis='both', labelsize=14)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


# Step 4d: Save the figure as PNG, PDF and TIF

fig_path     = r'C:\Users\cigde\Dropbox\Habituation_paper\October_Emily\final_figures\supplementary\suppl_fig_2'

os.makedirs(fig_path, exist_ok=True)

fig.savefig(os.path.join(fig_path, 'figure_suppl_2.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(fig_path, 'figure_suppl_2.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(fig_path, 'figure_suppl_2.tif'), dpi=300, bbox_inches='tight')


In [ ]:
## end